# `gelochip.gl` — API Reference

## Philosophy

| Concern | Old glayout | `gl` |
|---------|-------------|------|
| Placement | `prec_ref_center`, `movex`, `movey` | Automatic |
| Routing | `c_route`, `L_route`, `straight_route` | Automatic (`smart_route`) |
| Matching | `two_nfet_interdigitized` | Built into `gl.current_mirror` |
| Common-centroid | `common_centroid_ab_ba` | Built into `gl.diff_pair` |
| GuardRing | `tapring(...)` | Built into every FET (`with_tie=True`) |
| Ports | 200+ named ports | Clean terminal names (`g`, `d`, `s`, `ref`, `copy`, …) |
| Verification | Separate calls | `.drc()` `.lvs()` `.sim()` on result |

---

## Net — electrical wire
```python
vout = gl.Net('vout')   # named net
gl.vdd                  # VDD rail (singleton)
gl.gnd                  # GND rail (alias: gl.vss)
```

---

## Primitives — single devices

### NMOS / PMOS
```python
gl.nmos(w=2, l=None, fingers=1, multipliers=1,
         g=net, d=net, s=net, b=net,
         with_dummy=True, with_tie=True, sd_rmult=1)

gl.pmos(w=4, l=None, fingers=2,
         g=net, d=net, s=gl.vdd)
```

### Resistor
```python
gl.res(w=0.5, l=20, series=1, a=net_a, b=net_b)
```

### MIM Capacitor
```python
gl.mimcap(w=10, l=10, p=top_plate_net, n=bot_plate_net)
gl.mimcap_array(rows=2, cols=4, w=5, l=5, p=net, n=net)
```

### Via
```python
gl.via(layer1='met1', layer2='met3', net=vout)
```

### BJT
```python
gl.bjt(area=(5,5), bjt_type='pnp', b=vb, c=vc, e=ve)
```

---

## Pre-built Cells — optimal placement + routing built in

### Current Mirror (interdigitized)
```python
gl.current_mirror(
    w=4, l=None, ratio=2,
    n_or_p='n',           # 'n' = NMOS sink, 'p' = PMOS source
    ref=vbias,            # diode-connected reference
    copy=iout,            # mirror output
    rail=gl.gnd,          # source rail
)
```

### Differential Pair (common-centroid ABBA)
```python
gl.diff_pair(
    w=3, l=None, fingers=4,
    n_or_p='n',
    vp=vip,      # plus input
    vm=vim,      # minus input
    vtail=vtail, # tail current node
    vout_p=vop,  # plus-side drain
    vout_m=vom,  # minus-side drain
)
```

### Flipped Voltage Follower (FVF)
```python
gl.fvf(
    w_main=6.6, w_fb=3.7,
    n_or_p='n',
    vin=net, vout=net, ibias=net,
)
```

### Transmission Gate
```python
gl.transmission_gate(
    wp=2, wn=2,
    vin=net, vout=net, en=clk, enb=clkb,
)
```

### Stacked (Cascode) Mirror
```python
gl.stacked_cmirror(w=4, ratio=2, n_or_p='n',
                    ref=vbias, copy=iout, rail=gl.gnd)
```

---

## Module — reusable, parametric circuit block
```python
class MyCircuit(gl.Module):
    def __init__(self, w=4.0):
        super().__init__(gl.gf180)  # PDK
        self.w = w

    def layout(self):
        # Declare I/O pins
        vin  = self.pin('vin')
        vout = self.pin('vout')
        # Declare internal nets
        vmid = gl.Net('vmid')
        # Instantiate devices — connections declared here
        gl.pmos(w=self.w, g=vin, d=vout, s=gl.vdd)
        gl.nmos(w=self.w/2, g=vin, d=vout, s=gl.gnd)

chip = MyCircuit(w=4).build()
chip.show()    # render GDS
chip.drc()     # Magic DRC
chip.lvs()     # Netgen LVS
chip.sim()     # ngspice DC OP
chip.save('out.gds')
```

---

## Sequential — cascaded stages
```python
class TwoStage(gl.Sequential):
    def __init__(self):
        super().__init__(Stage1(), Stage2(), pdk=gl.gf180)

chip = TwoStage().build()
```

## Parallel — side-by-side matching
```python
vtail = gl.Net('vtail')
pair = gl.Parallel(
    gl.nmos(w=3, fingers=4, g=vp, s=vtail, d=vop),
    gl.nmos(w=3, fingers=4, g=vm, s=vtail, d=vom),
)
chip = pair.build()
```

---

## Functional API — quickest for one-offs
```python
vin, vout = gl.Net('vin'), gl.Net('vout')
mp = gl.pmos(w=4, g=vin, d=vout, s=gl.vdd)
mn = gl.nmos(w=2, g=vin, d=vout, s=gl.gnd)
chip = gl.build(mp, mn, name='inverter')
```

---

## Advanced — `gl.raw()` escape hatch
```python
# Access any glayout cell directly — no << >> needed
from glayout.cells.composite.fvf_based_ota.n_block import n_block
nb_comp = n_block(pdk, input_pair_params=(4, 2))

nb = gl.raw(
    nb_comp,
    connections={'inp': vip, 'inn': vim, 'gnd': gl.gnd},
    port_map={'inp': 'gate_inA', 'inn': 'gate_inB', 'gnd': 'cbias'},
)
chip = gl.build(nb, name='n_block')
```


In [ ]:
# Quick sanity check — all gl symbols load
import sys, os
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
sys.path.insert(0, os.path.abspath('../../src/gelochip'))
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel

print('PDK:', gl.gf180)
print('Primitives: nmos pmos res mimcap mimcap_array via bjt')
print('Cells:      current_mirror diff_pair fvf transmission_gate stacked_cmirror')
print('Module API: Module Sequential Parallel Layout')
print('Functional: build()')
print('Advanced:   raw()')
print()
print('Power rails:', gl.vdd, gl.gnd)
print('All OK.')